In [ ]:
import numpy as np
import torch
from torch import nn
from torch.autograd import grad
import torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
dataset = 'cf'
data_ = np.loadtxt('datasets/' + dataset + '/label.csv',skiprows=1,delimiter=',').astype(int)
truths = np.loadtxt('datasets/' + dataset + '/truth.csv',skiprows=1,delimiter=',').astype(int)[:,1]
idx = np.loadtxt('datasets/' + dataset + '/truth.csv',skiprows=1,delimiter=',').astype(int)[:,0]

truths = truths[np.argsort(idx)]
idx = np.sort(idx)
N = truths.size
M = data_[:,1].max()+1
L = data_[:,-1].max()+1

for i in np.arange(N):
    data_[data_[:,0]==idx[i],0] = -(i+1)

data_ = data_[data_[:,0]<0]
data_[:,0] = -data_[:,0]
data_[:,0] = data_[:,0] - 1

data = np.zeros((N,M,L))
for i in np.arange(data_.shape[0]):
    data[data_[i,0], data_[i,1], data_[i,-1]] = 1

data = torch.from_numpy(data).to(torch.float32).to(device)

data_flattened = data.reshape(-1, M*L)

delta = (data.sum(-1) > 0).to(torch.float32)[:,:,None]

MV_est = data.sum(1)/(data*delta).sum(-1).sum(-1)[:,None] + 1e-8

In [ ]:
def run_model(model,lambda_kl, lambda_s, lr):

    max_iters = 1000

    batch_size_train = 10
    batch_size_eval = 10
    n_val_samps = 1

    optim = torch.optim.Adam(model.parameters(), lr=lr)

    max_val_acc = 0
    max_val = np.inf

    for it in range(max_iters):
    
        recons, q_theta = model.forward(train_data.reshape(-1,M*L)[None,:,:].repeat(batch_size_train, 1, 1))
    
        q_theta = q_theta + 1e-8
    
        recon_term = (train_delta*train_data*torch.log(recons+1e-8)).sum()
        KL_error = (q_theta*(torch.log(q_theta) - torch.log(MV_est))).sum()
    
        weight_penalty = sum(p.abs().sum() for p in model.parameters())
        
        loss = -(recon_term - lambda_kl*KL_error) + lambda_s*weight_penalty
    
        loss.backward()
        optim.step()
        optim.zero_grad()    
        
        val_loss = 0
        theta_samps = np.zeros((n_val_samps,N,L))
        with torch.no_grad():
            for i in range(n_val_samps):
                recons, q_theta = model.forward(data.reshape(-1,M*L)[None,:,:].repeat(batch_size_train, 1, 1))
    
                valid_reconstruction = (val_delta*val_data*torch.log(recons+1e-8)).sum()
                val_loss -= valid_reconstruction/n_val_samps
                theta_samps[i] = q_theta[0].detach().cpu().numpy()
    
        acc = (theta_samps.sum(0).argmax(-1) == truths).mean()
    
        if val_loss < max_val:
            max_val = val_loss
            max_val_acc = acc
        
    return max_val, max_val_acc

In [ ]:
class LAA(nn.Module):
    def __init__(self, M, L):
        super().__init__()

        self.wp = nn.Linear(M*L, L)
        self.wq = nn.Linear(L, M*L)

    def forward(self,x):
        bs = x.shape[0]
        q_theta = nn.Softmax(dim=-1)(self.wp(x))
        samps = F.one_hot(torch.distributions.Categorical(probs=q_theta).sample(),num_classes=L).to(torch.float32)
        
        recons = nn.Softmax(dim=-1)(self.wq(samps).reshape(bs,-1, M, L))
        return recons, q_theta

class LAAL(nn.Module):
    def __init__(self, M, L, I):
        super().__init__()

        self.I = I
        self.wp = nn.Linear(M*L, L*I)
        self.wq = nn.Linear(L, M*L*I)
        self.wl = nn.Linear(M*L, I)

    def forward(self,x):
        bs = x.shape[0]

        z = nn.Softplus()(self.wl(x))
        
        q_theta = nn.Softmax(dim=-1)((self.wp(x).reshape((bs,-1,L,self.I))*z[:,:,None,:]).sum(-1))
        
        samps = F.one_hot(torch.distributions.Categorical(probs=q_theta).sample(),num_classes=L).to(torch.float32)

        recons = nn.Softmax(dim=-1)((self.wq(samps).reshape((bs,-1,M,L,self.I))*z[:,:,None,None,:]).sum(-1).reshape((bs,-1,M,L)))
        return recons, q_theta

In [ ]:
val_perc = .2
val_mask = torch.zeros(size=((N,M)), dtype=torch.bool, device=device)
nval = data.sum()*val_perc
count = 0
while count < nval:
    train_mask = (data.any(-1).to(int) - val_mask.to(int)).to(bool)
    score = torch.min((train_mask.sum(1)-1)[:,None], (train_mask.sum(0)-1)[None,:])*100
    med = score[train_mask.to(bool)][score[train_mask.to(bool)]>0].median()

    draw = torch.multinomial(((score*train_mask)/(score*train_mask).sum()).reshape(-1), 1)[0]
    val_mask.reshape(-1)[draw] = 1

    count += 1
    
val_data = torch.logical_and(data, val_mask[:,:,None]).to(torch.float32).to(device)
train_data = torch.logical_and(data, ~val_mask[:,:, None]).to(torch.float32).to(device)

val_delta = (val_data.sum(-1) > 0).to(torch.float32)[:,:,None]
train_delta = (train_data.sum(-1) > 0).to(torch.float32)[:,:,None]

In [ ]:
lambda_kls = [.0001, .001, .01, .1]
lambda_ss = [.0001, .001, .01, .1]
lrs = [.001, .01, .1]

nreps = 5
final_accs_b = np.zeros(nreps)
final_accs_o = np.zeros(nreps)
final_accs_l = np.zeros(nreps)

for r in range(nreps):
    print(f'rep {r}')
    
    val_losses_b = np.zeros((len(lambda_kls), len(lambda_ss), len(lrs)))
    accs_b = np.zeros((len(lambda_kls), len(lambda_ss), len(lrs)))

    val_losses_o = np.zeros((len(lambda_kls), len(lambda_ss), len(lrs)))
    accs_o = np.zeros((len(lambda_kls), len(lambda_ss), len(lrs)))

    val_losses_l = np.zeros((len(lambda_kls), len(lambda_ss), len(lrs)))
    accs_l = np.zeros((len(lambda_kls), len(lambda_ss), len(lrs)))
    
    for i in range(len(lambda_kls)):
        for j in range(len(lambda_ss)):
            for k in range(len(lrs)):
                model = LAA(M,L).to(device)
                val_losses_b[i,j,k], accs_b[i,j,k] = run_model(model,lambda_kls[i], lambda_ss[j], lrs[k])
                final_accs_b[r] = accs_b.reshape(-1)[np.argmin(val_losses_b)]
                
                model = LAAL(M,L,1).to(device)
                val_losses_o[i,j,k], accs_o[i,j,k] = run_model(model,lambda_kls[i], lambda_ss[j], lrs[k])
                final_accs_o[r] = accs_o.reshape(-1)[np.argmin(val_losses_o)]

                model = LAAL(M,L,2).to(device)
                val_losses_l[i,j,k], accs_l[i,j,k] = run_model(model,lambda_kls[i], lambda_ss[j], lrs[k])
                final_accs_l[r] = accs_l.reshape(-1)[np.argmin(val_losses_l)]

print(f'B {final_accs_b.mean():.3f}  {final_accs_b.std():.3f}')                                                       
print(f'O {final_accs_o.mean():.3f}  {final_accs_o.std():.3f}')
print(f'L {final_accs_l.mean():.3f}  {final_accs_l.std():.3f}')